# Flow Matching Variants and Practical Engineering

Once the core flow-matching pipeline is in place, the next questions are practical: how much the ODE solver matters, how rigid the linear path assumption is, how conditioning changes the transport, and how evaluation can avoid unnecessary repeated work.


A useful way to organize the practical discussion is around a few direct questions. How much does the ODE solver matter? How rigid is the linear path assumption? How can flow matching become **conditional** rather than unconditional? And how should repeated evaluation be organized so that **FID** and **KID** are informative without wasting compute?


```{important}
The main **engineering degrees of freedom** in flow matching are solver choice, path design, conditioning, and evaluation. Those choices often matter as much as the base loss itself.
```


## Solver Accuracy: Euler versus Midpoint

In continuous-time generators, **training the field** and **solving the field** are two different numerical tasks. Euler integration is the simplest possible solver, but it can be too crude when the learned velocity field varies noticeably over the interval. Midpoint or Heun-style updates often give a better approximation to the same trajectory for a modest extra cost.


In [ ]:
@torch.no_grad()
def sample_rectified_flow_euler(model, n_samples=16, steps=image_ode_steps, show_progress=True):
    model.eval()
    x = torch.randn(n_samples, image_channels, image_size, image_size, device=device)
    dt = 1.0 / steps
    iterator = range(steps)
    if show_progress:
        iterator = tqdm(iterator, desc="Euler sampling", leave=False)

    for i in iterator:
        t = torch.full((n_samples, 1), i / steps, device=device)
        x = x + dt * model(x, t)

    x = x.clamp(-1.0, 1.0)
    return 0.5 * (x + 1.0).cpu()


@torch.no_grad()
def sample_rectified_flow_midpoint(model, n_samples=16, steps=image_ode_steps, show_progress=True):
    model.eval()
    x = torch.randn(n_samples, image_channels, image_size, image_size, device=device)
    dt = 1.0 / steps
    iterator = range(steps)
    if show_progress:
        iterator = tqdm(iterator, desc="Midpoint sampling", leave=False)

    for i in iterator:
        t = torch.full((n_samples, 1), i / steps, device=device)
        v = model(x, t)
        x_mid = x + 0.5 * dt * v
        t_mid = torch.full((n_samples, 1), (i + 0.5) / steps, device=device)
        x = x + dt * model(x_mid, t_mid)

    x = x.clamp(-1.0, 1.0)
    return 0.5 * (x + 1.0).cpu()


The practical message is simple: if the field is reasonably smooth, **midpoint sampling** often gives visibly better images than Euler for the same step count. If samples are weak, the culprit may therefore be the **solver** rather than the training loop. This is one place where continuous-time models differ from VAEs and vanilla GANs: **numerical integration quality is part of generation quality**.


In [ ]:
euler_grid = utils.make_grid(sample_rectified_flow_euler(image_flow_model, n_samples=16), nrow=4, pad_value=1.0)
midpoint_grid = utils.make_grid(sample_rectified_flow_midpoint(image_flow_model, n_samples=16), nrow=4, pad_value=1.0)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(euler_grid.permute(1, 2, 0), cmap="gray")
axes[0].set_title("Euler")
axes[0].axis("off")

axes[1].imshow(midpoint_grid.permute(1, 2, 0), cmap="gray")
axes[1].set_title("Midpoint")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## Beyond the Linear Path

The straight interpolation path is attractive because its target velocity is extremely simple. But simplicity is not always the same as good geometry. One may prefer a path that injects extra Gaussian variability early on, or one that contracts uncertainty more carefully as time approaches one. This changes the **local supervision targets** and therefore changes what the network learns.


```{note}
A path is not just a preprocessing detail. In flow matching, the **path family defines the training signal**. Changing the path changes the geometry of the regression target seen by the model.
```


A minimal Gaussian-path version keeps the same mean interpolation but adds a time-dependent noise scale:
:::{math}
\boldsymbol{x}_t = \alpha_t \boldsymbol{x}_1 + (1-\alpha_t)\boldsymbol{x}_0 + \sigma_t \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N}(\boldsymbol{0}, I).
:::
The path is still analytically controlled, but it is no longer a deterministic line between endpoints. This can make the regression targets smoother or harder depending on the schedule design.


In [ ]:
def sample_gaussian_image_path(x1, sigma_min=0.01):
    x0 = torch.randn_like(x1)
    t = torch.rand(x1.size(0), 1, 1, 1, device=x1.device)
    alpha_t = t
    sigma_t = sigma_min + (1.0 - sigma_min) * (1.0 - t)
    eps = torch.randn_like(x1)
    xt = alpha_t * x1 + (1.0 - alpha_t) * x0 + sigma_t * eps

    # This target is a practical teaching approximation: it isolates the mean transport term.
    ut = x1 - x0
    return x0, x1, xt, ut, t.view(x1.size(0), 1)


def gaussian_path_flow_loss(model, x1):
    _, _, xt, ut, t = sample_gaussian_image_path(x1)
    pred = model(xt, t)
    return F.mse_loss(pred, ut)


This code is intentionally modest rather than maximalist. The key methodological point is that **changing the path changes the supervision**. In a more elaborate implementation one would derive and use the exact conditional velocity for the chosen path family.


## Conditional Flow Matching on FashionMNIST

Conditional generation is one of the most important extensions of the base model. Just as the conditional VAE and conditional GAN turn unconditional generation into **controlled generation**, a conditional flow model makes it possible to decide which class of object should emerge from the source noise. At a conceptual level, this is one of the clearest bridges from classical generative modeling to later **prompted generation** systems: the source randomness is no longer enough; an external conditioning signal shapes the transport.


On FashionMNIST the conditioning variable is just the class label. The methodological meaning is larger than that small example. The velocity field no longer answers only “where should mass move from here at this time?” It answers “where should mass move from here at this time **given the requested class**?” That is a first, simple form of controlled synthesis.


In [ ]:
num_classes = 10
cond_epochs = 25
cond_lr = 2e-4
class_names = fashion_train.classes


In [ ]:
class ConditionalImageVelocityUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, time_dim=128, num_classes=10):
        super().__init__()
        self.time_embedding = nn.Sequential(
            TimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.label_embedding = nn.Embedding(num_classes, time_dim)
        self.input_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.down1 = ImageResidualBlock(base_channels, base_channels, time_dim)
        self.downsample1 = nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.down2 = ImageResidualBlock(base_channels * 2, base_channels * 2, time_dim)
        self.downsample2 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.down3 = ImageResidualBlock(base_channels * 4, base_channels * 4, time_dim)
        self.mid = ImageResidualBlock(base_channels * 4, base_channels * 4, time_dim)
        self.upsample2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.up2 = ImageResidualBlock(base_channels * 4, base_channels * 2, time_dim)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.up1 = ImageResidualBlock(base_channels * 2, base_channels, time_dim)
        self.output_conv = nn.Conv2d(base_channels, in_channels, kernel_size=1)

    def forward(self, x, t, labels):
        t_emb = self.time_embedding(t)
        label_emb = self.label_embedding(labels)
        cond_emb = t_emb + label_emb
        x0 = self.input_conv(x)
        x1 = self.down1(x0, cond_emb)
        x2 = self.downsample1(x1)
        x2 = self.down2(x2, cond_emb)
        x3 = self.downsample2(x2)
        x3 = self.down3(x3, cond_emb)
        x_mid = self.mid(x3, cond_emb)
        x_up = self.upsample2(x_mid)
        x_up = torch.cat([x_up, x2], dim=1)
        x_up = self.up2(x_up, cond_emb)
        x_up = self.upsample1(x_up)
        x_up = torch.cat([x_up, x1], dim=1)
        x_up = self.up1(x_up, cond_emb)
        return self.output_conv(x_up)


conditional_flow_model = ConditionalImageVelocityUNet(
    in_channels=image_channels,
    base_channels=image_base_channels,
    time_dim=image_time_dim,
    num_classes=num_classes,
).to(device)
conditional_optimizer = torch.optim.AdamW(conditional_flow_model.parameters(), lr=cond_lr, weight_decay=1e-4)


def conditional_rectified_flow_loss(model, x1, labels):
    _, _, xt, ut, t = sample_image_path(x1)
    pred = model(xt, t, labels)
    return F.mse_loss(pred, ut)


The only new information entering the model is the label embedding, but conceptually that changes the whole role of the network. It is no longer learning one class-agnostic transport law. It is learning a **family** of class-specific transport behaviors that share parameters but differ through conditioning.


In [ ]:
conditional_history = []

for epoch in tqdm(range(cond_epochs), desc="Conditional flow epochs"):
    conditional_flow_model.train()
    running_loss = 0.0

    for x1, labels in tqdm(fashion_loader, desc="train", leave=False):
        x1 = x1.to(device)
        labels = labels.to(device)
        conditional_optimizer.zero_grad()
        loss = conditional_rectified_flow_loss(conditional_flow_model, x1, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(conditional_flow_model.parameters(), max_norm=1.0)
        conditional_optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(fashion_loader)
    conditional_history.append(epoch_loss)
    print(f"Epoch {epoch + 1:02d} | conditional rectified-flow loss: {epoch_loss:.6f}")


In [ ]:
@torch.no_grad()
def sample_conditional_rectified_flow(model, labels, steps=image_ode_steps, show_progress=True):
    model.eval()
    labels = labels.to(device)
    n_samples = labels.size(0)
    x = torch.randn(n_samples, image_channels, image_size, image_size, device=device)
    dt = 1.0 / steps
    iterator = range(steps)
    if show_progress:
        iterator = tqdm(iterator, desc="conditional sampling", leave=False)

    for i in iterator:
        t = torch.full((n_samples, 1), i / steps, device=device)
        v = model(x, t, labels)
        x_mid = x + 0.5 * dt * v
        t_mid = torch.full((n_samples, 1), (i + 0.5) / steps, device=device)
        x = x + dt * model(x_mid, t_mid, labels)

    x = x.clamp(-1.0, 1.0)
    return 0.5 * (x + 1.0).cpu()


label_grid = torch.arange(num_classes).repeat_interleave(4)
conditional_samples = sample_conditional_rectified_flow(conditional_flow_model, label_grid, show_progress=False)
conditional_grid = utils.make_grid(conditional_samples, nrow=4, pad_value=1.0)
plt.figure(figsize=(8, 12))
plt.imshow(conditional_grid.permute(1, 2, 0), cmap="gray")
plt.axis("off")
plt.show()


The expected behavior is stronger than in the unconditional case. It is not enough to generate generic clothing-like blobs. The row conditioned on “Sandal” should drift toward sandal-like shapes, the row conditioned on “Bag” should drift toward bag-like shapes, and so on. Conditional flow matching is therefore a useful precursor to the broader modern story of externally guided generation.


## Efficient FID and KID with Cached Real Features

When several solvers or several variants of the same model are being compared, repeatedly recomputing the real-data features becomes wasteful. The metrics below accumulate the real features once and then reuse them across several generators or samplers.


In [ ]:
@torch.no_grad()
def build_real_metric_cache(real_loader, device, num_samples=1000, metric_batch_size=32):
    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True,
        reset_real_features=False,
    ).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(
        feature=2048,
        subsets=10,
        subset_size=100,
        normalize=True,
        reset_real_features=False,
    ).to(device)

    seen_real = 0
    for real_images, _ in tqdm(real_loader, desc="Caching real features", leave=False):
        remaining = num_samples - seen_real
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = 0.5 * (real_images + 1.0)
        real_images = prepare_for_inception_metrics(real_images)
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen_real += real_images.size(0)

    return fid, kid


@torch.no_grad()
def evaluate_cached_flow_sampler(fid, kid, sampler_fn, device, num_samples=1000, metric_batch_size=32):
    # Create fresh metric objects that reuse the already-computed real statistics.
    eval_fid = fid.clone().to(device)
    eval_kid = kid.clone().to(device)

    generated = 0
    pbar = tqdm(total=num_samples, desc="Cached fake metrics", leave=False)
    while generated < num_samples:
        batch_n = min(metric_batch_size, num_samples - generated)
        fake_images = sampler_fn(batch_n).to(device)
        fake_images = prepare_for_inception_metrics(fake_images)
        eval_fid.update(fake_images, real=False)
        eval_kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()

    kid_mean, kid_std = eval_kid.compute()
    return {
        "fid": eval_fid.compute().item(),
        "kid_mean": kid_mean.item(),
        "kid_std": kid_std.item(),
    }


The exact cloning behavior depends on the metric implementation version, so in a production codebase it is worth verifying that the cached real statistics are being reused as intended. The broader pattern is simple: separate the expensive real-data pass from the repeated fake-sample evaluations.


## Practical Failure Modes

Flow matching tends to have a stable objective, but that does not mean every run is equally good. Several problems can still appear:

- The vector field can be accurate on the training interpolation distribution but less accurate on its own sampled trajectories.
- The solver can be too coarse even when the training loss looks healthy.
- The path can be geometrically awkward, making the regression target harder than it needed to be.
- The image model can be underpowered, especially if the target distribution has many fine-grained modes.

In practice, it is useful to inspect samples, compare solvers, monitor the loss, and be explicit about the **path design**.


## Closing Perspective

Flow matching is not just one loss attached to one straight-line path. It is a flexible family of deterministic transport models whose behavior depends on the **conditional path**, the **conditioning mechanism**, the **architecture**, and the **ODE solver**.
